# Жива перевірка чекпоінта Whisper

1. Завантажує потрібний `.ckpt` зі віддаленого сервера (`rsync` через SSH, з докачуванням).
2. Завантажує модель локально (ваги `model.*` поверх базового `openai/whisper-medium`, як у `evaluation/evaluate.py`).
3. Записує аудіо з мікрофона в браузері й одразу розпізнає його. Можна також завантажити аудіофайл.

Запис іде через мікрофон браузера (`MediaRecorder`), тому працює і у WSL. Браузер має відкривати Jupyter за `localhost`, інакше мікрофон заблокується.

Потрібно один раз: `pip install anywidget` (решта вже є в `requirements.txt`), а також `ffmpeg` у системі.

## 1. Налаштування

In [ ]:
import os
from pathlib import Path

# Секрети (хост, порт, шлях) лежать у .env в корені репозиторію, який не комітиться.
# Потрібні ключі: WHISPER_SSH_HOST, WHISPER_SSH_PORT (необов'язково), WHISPER_REMOTE_CKPT.
def load_env() -> None:
    for root in [Path.cwd(), *Path.cwd().parents]:
        env_file = root / ".env"
        if env_file.is_file():
            for raw in env_file.read_text(encoding="utf-8").splitlines():
                line = raw.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
            return
    raise SystemExit(".env не знайдено")


load_env()
SSH_HOST = os.environ["WHISPER_SSH_HOST"]
SSH_PORT = int(os.environ.get("WHISPER_SSH_PORT", "22"))
REMOTE_CKPT = os.environ["WHISPER_REMOTE_CKPT"]

# --- Локально ---
CACHE_DIR = Path.home() / ".cache" / "asr_ssl_checkpoints"
BASE_MODEL = "openai/whisper-medium"   # model.id з config.yaml
LANGUAGE = "uk"                        # model.language з config.yaml
DEVICE = "cuda"                        # "cuda" або "cpu"; якщо CUDA немає, буде cpu


## 2. Завантаження чекпоінта

In [ ]:
import subprocess

CACHE_DIR.mkdir(parents=True, exist_ok=True)
local_ckpt = CACHE_DIR / Path(REMOTE_CKPT).name

# --partial докачує перервані завантаження; повторний запуск пропускає вже готовий файл.
subprocess.run(
    ["rsync", "-avP", "--partial", "-e", f"ssh -p {SSH_PORT}", f"{SSH_HOST}:{REMOTE_CKPT}", str(local_ckpt)],
    check=True,
)
print(local_ckpt, f"{local_ckpt.stat().st_size / 1e9:.2f} GB")


## 3. Завантаження моделі

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

device = DEVICE if torch.cuda.is_available() else "cpu"
processor = WhisperProcessor.from_pretrained(BASE_MODEL)
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)

checkpoint = torch.load(local_ckpt, map_location="cpu", weights_only=False)
state = checkpoint.get("state_dict", checkpoint)
weights = {k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")}
assert weights, f"У {local_ckpt} немає ваг model.*"
model.load_state_dict(weights)
del checkpoint, state, weights

model = model.to(device).eval()
if device == "cuda":
    model = model.half()
print("модель готова на", device)


## 4. Розпізнавання

In [ ]:
import subprocess
import numpy as np

SR = 16000


def load_audio(data: bytes) -> np.ndarray:
    """Будь-який формат (webm/opus із браузера, wav, mp3...) -> float32 mono 16 кГц через ffmpeg."""
    out = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", "pipe:0", "-f", "f32le", "-ac", "1", "-ar", str(SR), "pipe:1"],
        input=data, capture_output=True, check=True,
    ).stdout
    return np.frombuffer(out, dtype=np.float32)


@torch.inference_mode()
def transcribe(audio: np.ndarray) -> str:
    # До 30 с — один прохід; довше ріжемо на шматки по 30 с.
    chunks = [audio[i : i + 30 * SR] for i in range(0, len(audio), 30 * SR)]
    texts = []
    for chunk in chunks:
        features = processor.feature_extractor(chunk, sampling_rate=SR, return_tensors="pt").input_features
        features = features.to(device, dtype=model.dtype)
        ids = model.generate(features, language=LANGUAGE, task="transcribe", num_beams=1)
        texts.append(processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip())
    return " ".join(t for t in texts if t)


## 5. Запис з мікрофона
Натисніть «Запис», говоріть, натисніть «Стоп» — текст з'явиться нижче одразу.

In [ ]:
import anywidget
import traitlets
import ipywidgets as widgets
from IPython.display import display


class Recorder(anywidget.AnyWidget):
    _esm = """
    export default {
      render({ model, el }) {
        const btn = document.createElement("button");
        btn.textContent = "● Запис";
        btn.style.cssText = "padding:8px 18px;font-size:15px;cursor:pointer";
        const status = document.createElement("span");
        status.style.marginLeft = "12px";
        el.append(btn, status);
        let rec = null, chunks = [], stream = null;

        btn.onclick = async () => {
          if (rec) { rec.stop(); return; }
          try {
            stream = await navigator.mediaDevices.getUserMedia({ audio: true });
          } catch (e) { status.textContent = "Немає доступу до мікрофона: " + e.message; return; }
          chunks = [];
          rec = new MediaRecorder(stream);
          rec.ondataavailable = (e) => chunks.push(e.data);
          rec.onstop = async () => {
            stream.getTracks().forEach((t) => t.stop());
            const blob = new Blob(chunks, { type: rec.mimeType });
            rec = null;
            btn.textContent = "● Запис";
            status.textContent = "обробка...";
            const buf = new Uint8Array(await blob.arrayBuffer());
            model.set("audio", new DataView(buf.buffer));
            model.save_changes();
          };
          rec.start();
          btn.textContent = "■ Стоп";
          status.textContent = "іде запис...";
        };
        model.on("change:status", () => { status.textContent = model.get("status"); });
      },
    };
    """
    audio = traitlets.Bytes(b"").tag(sync=True)
    status = traitlets.Unicode("").tag(sync=True)


recorder = Recorder()
out = widgets.Output()
last_audio = {"data": None}


def on_audio(change):
    data = bytes(change["new"])
    if not data:
        return
    with out:
        out.clear_output()
        try:
            audio = load_audio(data)
            last_audio["data"] = audio
            seconds = len(audio) / SR
            display(__import__("IPython").display.Audio(audio, rate=SR))
            text = transcribe(audio)
            print(f"[{seconds:.1f} с]  {text}")
        except Exception as e:
            print("Помилка:", e)
    recorder.status = "готово"


recorder.observe(on_audio, names="audio")
display(recorder, out)


## 6. Альтернатива: аудіофайл

In [ ]:
upload = widgets.FileUpload(accept="audio/*,video/*", multiple=False, description="Файл")
file_out = widgets.Output()


def on_upload(change):
    items = upload.value
    if not items:
        return
    item = items[0] if isinstance(items, (list, tuple)) else next(iter(items.values()))
    with file_out:
        file_out.clear_output()
        audio = load_audio(bytes(item["content"]))
        print(f"[{len(audio) / SR:.1f} с]  {transcribe(audio)}")


upload.observe(on_upload, names="value")
display(upload, file_out)
